# General information
<span style="color: green"> **Please use the BIDS structure**</span> so that pipeline can run smoothly. Place ipynb-notebooks on the top level of the project folder (same level as subject folders).


**Gereral processing steps:**  

0. [Import libraries](#0-import-libraries)  
1. [Set channel types and montage](#1-channel-types-montage)  
2. [Filter](#2-filter)  
3. [Restoring Reference channel](#3-restoring-reference-channel-and-average-reference)  
4. [Epoching](#4-epoching) 
5. [GEDAI](#5-gedai)  
6. [Adaptive Mixture Independent Component Analysis](#6-adaptive-mixture-independent-component-analysis)  

::: {#eeg-process}

![](Fig2-EEGProcess.svg){width=80%}

EEG processing overview

:::

# 0. Import libraries

The analysis pipeline is based on the following libraries. In case of an error in the execution of this cell, probably one or more of the libraries is not installed. In this case, start a terminal in **ANACONDA.NAVIGATOR** *Environments>Terminal* and install the library in question using the command <span style='color: red'>*pip install [library name]*</span>.

In [1]:
import warnings                 # switch of pandas and mne warnings
warnings.simplefilter(action='ignore')

import requests
import os
import os.path as op
import time
from pathlib import Path

import mne

import gc
import numpy as np
import pandas as pd
import ipyfilechooser
import ipywidgets as widgets
import matplotlib.pyplot as plt
import seaborn as sns
import datetime
import pyprep
from pyprep.prep_pipeline import PrepPipeline
from datetime import datetime, timedelta
from datetime import datetime, timezone
#from playsound import playsound
from openpyxl import load_workbook
from mne.export import export_raw

from collections import defaultdict
from mne.datasets import fetch_fsaverage
from mne.evoked import combine_evoked
from mne.forward import make_forward_dipole
from mne.simulation import simulate_evoked
from mne.viz import circular_layout
from mne_connectivity import spectral_connectivity_epochs
from mne_connectivity.viz import plot_connectivity_circle
from mne_icalabel import label_components
from mne_icalabel.gui import label_ica_components
#from asrpy import ASR
from meegkit.asr import ASR
from gedai import Gedai
from gedai.viz import plot_mne_style_overlay_interactive

from nilearn.image import index_img
from nilearn.plotting import plot_stat_map
from mne import read_evokeds
from mne.datasets import sample
from mne.minimum_norm import  make_inverse_operator, apply_inverse, read_inverse_operator, apply_inverse_epochs

from pyamica import AMICA, AmicaICA
import torch

%matplotlib qt

mne.set_log_level("ERROR")  # only show errors (no warnings or information messages)

# 1. Channel types, montage

## 1.1. Load files 
Find and load the concatenated eeg files for further processing.

In [4]:
wd = Path.cwd()
wd.parent

PosixPath('/home/joel/Documents/BIDS - Kopie/CoMoCut_Proof-of-conept')

In [ ]:
wd = Path.cwd()
CONCAT_RAW_PATH = wd / "derivatives" / "02_concat-raw"

# -------------------------------------------------------------------------
# LOAD CONCATENATED RAW FILES
# Discovers all .fif files under 02_concat-raw/sub-XX/ses-XX/eeg/
# -------------------------------------------------------------------------
raw_files = sorted(CONCAT_RAW_PATH.rglob("*desc-concat_eeg.fif"))
print(f"Found {len(raw_files)} concatenated raw files")

eegConcat = []

for fif_file in raw_files:
    ses_name = fif_file.parent.parent.name
    sub_name = fif_file.parent.parent.parent.name

    try:
        raw = mne.io.read_raw_fif(fif_file, preload=True)
        eegConcat.append(raw)
        print(f"Loaded : {sub_name} / {ses_name}")

    except Exception as e:
        print(f"Error loading {sub_name} / {ses_name} : {e}")

Found 2 concatenated raw files
Loaded : sub-01 / ses-01
Loaded : sub-02 / ses-01


## 1.2. Set channel types and montage

In [6]:
# -------------------------------------------------------------------------
# SET CHANNEL TYPES AND MONTAGE
# Applied to all concatenated raw files.
# Channel types distinguish EEG from EOG, EMG, and acceleration sensor channels.
# EasyCap M1 montage provides standard electrode locations.
# -------------------------------------------------------------------------

for raw in eegConcat:
    # Set non-EEG channel types
    raw.set_channel_types({
        "HEOG"   : "eog",
        "VEOG"   : "eog",
        "NeckEMG": "emg",
        "x_dir"  : "misc",
        "y_dir"  : "misc",
        "z_dir"  : "misc",
    })

    # Apply standard EasyCap M1 montage for electrode locations
    raw.set_montage(mne.channels.make_standard_montage("easycap-M1"))

print(f"Channel types and montage set for {len(eegConcat)} files")

Channel types and montage set for 2 files


# 2. Filter

Originally, we used a 1 Hz to 40 Hz band-pass filter.
For future analyses, use only 1 Hz high-pass filter. Then apply 40 Hz low-pass at the end of the source-space processing (after AMICA).

In [ ]:
# -------------------------------------------------------------------------
# BANDPASS FILTER
# Current study: 1–40 Hz bandpass.
# Note: for future studies a 1 Hz high-pass only is recommended,
# leaving the low-pass to be applied after all other 
# source-space processing (after AMICA if needed).
# To switch, set h_freq=None.
# -------------------------------------------------------------------------
L_FREQ = 1    # high-pass cutoff (Hz)
H_FREQ = None   # low-pass cutoff (Hz) — set to None for high-pass only

for raw in eegConcat:
    raw.filter(l_freq=L_FREQ, h_freq=H_FREQ)
    print(f"Filtered : {Path(raw.filenames[0]).name}")

print(f"\nFilter applied : {L_FREQ}–{H_FREQ} Hz" if H_FREQ else f"\nHigh-pass filter applied : {L_FREQ} Hz")

Filtered : sub-01_ses-01_task-sidecut_desc-concat_eeg.fif
Filtered : sub-02_ses-01_task-sidecut_desc-concat_eeg.fif

High-pass filter applied : 0.5 Hz


# 3. Restoring reference channel and average reference

Restore reference-channel (FCz) as zero-channel. Then, use average reference.

Finally, safe filtered EEG files before pyPREP and ASR.

In [8]:
# -------------------------------------------------------------------------
# FUNCTION: add_ref
# Adds FCz back as a zero channel (online recording reference) and
# re-references to average. The montage is re-applied after adding FCz
# since adding a new channel resets electrode locations.
#
# Parameters
# ----------
# raw : mne.io.Raw — filtered raw EEG data without FCz
#
# Returns
# -------
# raw : mne.io.Raw — re-referenced raw EEG data with FCz restored
# -------------------------------------------------------------------------
def add_ref(raw):
    if "FCz" not in raw.ch_names:
        raw = mne.add_reference_channels(raw, ref_channels=["FCz"])
    else:
        print("FCz already present, skipping")

    raw.set_montage(mne.channels.make_standard_montage("easycap-M1"))
    raw.set_eeg_reference("average", projection=False)
    return raw

In [9]:
# -------------------------------------------------------------------------
# APPLY REFERENCE TO ALL FILES
# -------------------------------------------------------------------------
eegConcat = [add_ref(raw) for raw in eegConcat]
print(f"FCz added and average reference applied to {len(eegConcat)} files")

FCz added and average reference applied to 2 files


# 4. Epoching

In [10]:
# -------------------------------------------------------------------------
# EPOCH CONFIGURATION
# Each entry corresponds to one epoch type (RS_left, RS_right, IC_left, IC_right).
# -------------------------------------------------------------------------
scal = {"eeg": 1e-4, "eog": 1e-4, "emg": 1e2, "misc": 1e3}

epoch_names  = ["RS_left", "RS_right", "IC_left", "IC_right"]
epoch_range  = [(-4.5, 0.5), (-4.5, 0.5), (-5.0, 0.0), (-5.0, 0.0)]
epoch_base   = [(-4.5, -2.5), (-4.5, -2.5), (-5.0, -3.0), (-5.0, -3.0)]
epoch_reject = [(-1.5, -1.0), (-1.5, -1.0), (-2.0, -1.5), (-2.0, -1.5)]

In [11]:
# -------------------------------------------------------------------------
# FUNCTION: make_epochs
# Creates epochs for a single annotation type from a Raw object.
#
# Parameters
# ----------
# raw  : mne.io.Raw — preprocessed raw data
# i    : int — index into epoch_names/range/base/reject lists
#
# Returns
# -------
# epochs : mne.Epochs
# -------------------------------------------------------------------------
def make_epochs(raw, i):
    evt, _ = mne.events_from_annotations(raw, event_id={epoch_names[i]: 1001})

    epochs = mne.Epochs(
        raw,
        events    = evt,
        event_id  = {epoch_names[i]: 1001},
        tmin      = epoch_range[i][0],
        tmax      = epoch_range[i][1],
        baseline  = epoch_base[i],
        #reject    = {"eeg": 500e-6},
        flat      = {"eeg": 1e-10},
        detrend   = 1,
        preload   = True,
    )
    return epochs

In [12]:
# -------------------------------------------------------------------------
# EPOCHING LOOP
# Uses already-loaded raw files from eegConcat, creates epochs for all four
# annotation types, and stores them in lists for metadata attachment.
# Files are not saved here — metadata is added in the next step.
# -------------------------------------------------------------------------

# Storage lists — one entry per subject/session
epRSL = []   # RS_left epochs
epRSR = []   # RS_right epochs
epICL = []   # IC_left epochs
epICR = []   # IC_right epochs

sub_ses_list = []  # track (sub_name, ses_name) in same order as epoch lists

for raw in eegConcat:
    fif_path = Path(raw.filenames[0])
    ses_name = fif_path.parent.parent.name
    sub_name = fif_path.parent.parent.parent.name

    print(f"\nEpoching : {sub_name} / {ses_name}")

    try:
        for ep_list, i, label in zip(
            [epRSL, epRSR, epICL, epICR],
            [0, 1, 2, 3],
            ["RS_left", "RS_right", "IC_left", "IC_right"]
        ):
            epochs = make_epochs(raw, i)
            n_total    = len(epochs.drop_log)  # before rejection
            n_kept     = len(epochs)           # after rejection
            n_rejected = n_total - n_kept
            ep_list.append(epochs)
            print(f"  {label:<12} : {n_kept} kept / {n_rejected} rejected / {n_total} total")

        sub_ses_list.append((sub_name, ses_name))

    except Exception as e:
        print(f"  Error : {sub_name} / {ses_name} : {e}")


Epoching : sub-01 / ses-01
  RS_left      : 58 kept / 0 rejected / 58 total
  RS_right     : 65 kept / 0 rejected / 65 total
  IC_left      : 58 kept / 0 rejected / 58 total
  IC_right     : 65 kept / 0 rejected / 65 total

Epoching : sub-02 / ses-01
  RS_left      : 42 kept / 0 rejected / 42 total
  RS_right     : 51 kept / 0 rejected / 51 total
  IC_left      : 42 kept / 0 rejected / 42 total
  IC_right     : 51 kept / 0 rejected / 51 total


## 4.2. Add ATR

In [13]:
# -------------------------------------------------------------------------
# FUNCTION: filter_by_atr
# Rejects epochs with ATR exceeding ATR_MAX. If ATR_MAX is None, the
# threshold is computed as mean + 3 SD of the ATR distribution per subject.
# Metadata is updated to reflect only the retained epochs.
#
# Parameters
# ----------
# epochs  : mne.Epochs — epoch object with ATR column in metadata
# atr_max : float | None — upper ATR bound in ms (None = mean + 3 SD)
#
# Returns
# -------
# filtered_epochs : mne.Epochs — epochs with ATR <= atr_max
# -------------------------------------------------------------------------
def filter_by_atr(epochs, atr_max):
    if epochs.metadata is None or "ATR" not in epochs.metadata.columns:
        raise ValueError("No ATR metadata found in epochs")

    atr_values = epochs.metadata["ATR"].values

    # Resolve None threshold from data distribution (mean + 3 SD)
    hi = (np.mean(atr_values) + 3 * np.std(atr_values)) if atr_max is None else atr_max

    keep_idx   = np.where(atr_values <= hi)[0]
    n_rejected = len(epochs) - len(keep_idx)

    # Handle None filename for in-memory epochs (not yet saved to disk)
    fname = Path(epochs.filename).name if epochs.filename is not None else "in-memory"

    print(f"  {fname} : "
          f"ATR <= {hi:.0f} ms → "
          f"{len(keep_idx)} kept / {n_rejected} rejected")

    filtered          = epochs[keep_idx]
    filtered.metadata = filtered.metadata.reset_index(drop=True)

    return filtered

In [ ]:
# -------------------------------------------------------------------------
# ADD ATR METADATA, FILTER LONG ATR TRIALS, AND SAVE EPOCHS
# ATR is computed from RS and IC onsets and attached as metadata first
# (needed to know which trials to reject), but the reported ATR summary
# statistics reflect only the RETAINED trials, after rejection.
# -------------------------------------------------------------------------
EPOCHS_PATH = wd / "derivatives" / "03_epochs"
ATR_MAX     = 1500  # ms — trials with ATR > this value are rejected

for i, (sub_name, ses_name) in enumerate(sub_ses_list):
    print(f"\nAttaching metadata : {sub_name} / {ses_name}")

    output_dir = EPOCHS_PATH / sub_name / ses_name / "eeg"
    output_dir.mkdir(parents=True, exist_ok=True)

    for epochs, label in zip(
        [epRSL[i], epRSR[i], epICL[i], epICR[i]],
        ["RS_left", "RS_right", "IC_left", "IC_right"]
    ):
        try:
            side = label.split("_")[1]

            # Determine RS and IC epoch pair for ATR computation
            if label in ("RS_left", "IC_left"):
                rs_epochs = epRSL[i]
                ic_epochs = epICL[i]
            else:
                rs_epochs = epRSR[i]
                ic_epochs = epICR[i]

            # Compute ATR (needed before rejection can be applied)
            rs_onsets = rs_epochs.events[:, 0] / rs_epochs.info["sfreq"]
            ic_onsets = ic_epochs.events[:, 0] / ic_epochs.info["sfreq"]

            if len(rs_onsets) == len(ic_onsets):
                atr_ms = (ic_onsets - rs_onsets) * 1000
            else:
                print(f"  Warning : ATR length mismatch for {label} "
                      f"({len(rs_onsets)} RS vs {len(ic_onsets)} IC) — filling with NaN")
                atr_ms = np.full(len(epochs), np.nan)

            meta = pd.DataFrame({
                "trial": np.arange(1, len(epochs) + 1),
                "ATR"  : atr_ms,
                "side" : side,
            })
            epochs.metadata = meta

            # -----------------------------------------------------------------
            # REJECT LONG ATR TRIALS
            # -----------------------------------------------------------------
            n_before = len(epochs)
            epochs   = filter_by_atr(epochs, atr_max=ATR_MAX)
            epochs.metadata = epochs.metadata.reset_index(drop=True)
            n_after  = len(epochs)
            n_rejected = n_before - n_after

            print(f"  {label:<10} : {n_rejected} / {n_before} trials rejected "
                  f"(ATR > {ATR_MAX} ms), {n_after} retained")

            if n_after == 0:
                print(f"  Warning : no epochs remaining after ATR filter for {label} — skipping")
                continue

            # -----------------------------------------------------------------
            # REPORT ATR SUMMARY ON RETAINED TRIALS ONLY
            # -----------------------------------------------------------------
            atr_retained = epochs.metadata["ATR"].values
            print(f"  ATR {side:<6} (retained) : "
                  f"n={n_after} | "
                  f"mean={np.nanmean(atr_retained):.1f} ± {np.nanstd(atr_retained):.1f} ms | "
                  f"range=[{np.nanmin(atr_retained):.1f}, {np.nanmax(atr_retained):.1f}] ms")

            # Save
            output_name = f"{sub_name}_{ses_name}_task-sidecut_{label}-epo.fif"
            output_path = output_dir / output_name
            epochs.save(output_path, overwrite=True)
            print(f"  Saved : {output_name}")

        except Exception as e:
            print(f"  Error saving {label} : {sub_name} / {ses_name} : {e}")


Attaching metadata : sub-01 / ses-01
  in-memory : ATR <= 1500 ms → 58 kept / 0 rejected
  RS_left    : 0 / 58 trials rejected (ATR > 1500 ms), 58 retained
  ATR left   (retained) : n=58 | mean=771.3 ± 88.9 ms | range=[552.0, 934.0] ms
  Saved : sub-01_ses-01_task-sidecut_RS_left-epo.fif
  in-memory : ATR <= 1500 ms → 65 kept / 0 rejected
  RS_right   : 0 / 65 trials rejected (ATR > 1500 ms), 65 retained
  ATR right  (retained) : n=65 | mean=764.1 ± 86.3 ms | range=[624.0, 1022.0] ms
  Saved : sub-01_ses-01_task-sidecut_RS_right-epo.fif
  in-memory : ATR <= 1500 ms → 58 kept / 0 rejected
  IC_left    : 0 / 58 trials rejected (ATR > 1500 ms), 58 retained
  ATR left   (retained) : n=58 | mean=771.3 ± 88.9 ms | range=[552.0, 934.0] ms
  Saved : sub-01_ses-01_task-sidecut_IC_left-epo.fif
  in-memory : ATR <= 1500 ms → 65 kept / 0 rejected
  IC_right   : 0 / 65 trials rejected (ATR > 1500 ms), 65 retained
  ATR right  (retained) : n=65 | mean=764.1 ± 86.3 ms | range=[624.0, 1022.0] ms
  Sa

# 5. GEDAI

Use the Generalized Eigenvalue De-Artifacting Instrument (GEDAI) from [@rosReturnGEDAIUnsupervised2025].

First use the general GEDAI. Then, use the spectral version for more aggressive and accurate cleaning.

Link:
- *GEDAI*: [https://github.com/neurotuning/GEDAI-master](https://github.com/neurotuning/GEDAI-master)

## 5.1. GEDAI

In [ ]:
EPOCHS_PATH     = wd / "derivatives" / "03_epochs"
GEDAI_PATH = wd / "derivatives" / "04a_epochs_gedai"

# -------------------------------------------------------------------------
# CONFIGURATION
# Choose which epoch types to fit AMICA on.
# "IC"  — IC_left and IC_right only
# "RS"  — RS_left and RS_right only
# None  — all four epoch types
# -------------------------------------------------------------------------
EPOCH_TYPES = None  # "IC", "RS", or None to process all epochs

if EPOCH_TYPES == "IC":
    epoch_pattern = "*IC_*-epo.fif"
elif EPOCH_TYPES == "RS":
    epoch_pattern = "*RS_*-epo.fif"
elif EPOCH_TYPES == None:
    epoch_pattern = "*-epo.fif"
else:
    raise ValueError(f"Unknown AMICA_EPOCH_TYPES: '{AMICA_EPOCH_TYPES}'. Choose 'IC', 'RS', or None.")

# -------------------------------------------------------------------------
# CONFIGURATION
# GEDAI expects EEG-only channels — its leadfield reference covariance is
# built from 10-5 system electrode names, so non-EEG channels (EOG, EMG,
# misc) must be excluded before fitting/transforming, then reattached
# afterward.
# Output: derivatives/04_epochs_gedai/sub-XX/ses-XX/eeg/
# -------------------------------------------------------------------------
#GEDAI_DURATION         = 2.0
#GEDAI_OVERLAP          = 0.5
#GEDAI_REJECT_BY_ANNOT  = False
GEDAI_REFERENCE_COV    = "leadfield"
GEDAI_SENSAI_METHOD    = "gridsearch"
GEDAI_NOISE_MULTIPLIER = 3.0
# -------------------------------------------------------------------------
gedai_files = sorted(EPOCHS_PATH.rglob(epoch_pattern))
print(f"Found {len(gedai_files)} epoch files to process (EPOCH_TYPES='{EPOCH_TYPES}')")

for fif_path in gedai_files:
    ses_name = fif_path.parent.parent.name
    sub_name = fif_path.parent.parent.parent.name

    for candidate in ["IC_left", "IC_right", "RS_left", "RS_right"]:
        if candidate in fif_path.name:
            label = candidate
            break
    else:
        print(f"  Could not determine label from filename, skipping : {fif_path.name}")
        continue

    print(f"\nGEDAI : {sub_name} / {ses_name} / {label}")

    try:
        e = mne.read_epochs(fif_path, preload=True)

        e_eeg = e.copy().pick("eeg")
        non_eeg_picks = mne.pick_types(e.info, eeg=False, eog=True, emg=True, misc=True)
        e_non_eeg = e.copy().pick(non_eeg_picks) if len(non_eeg_picks) > 0 else None

        gedai = Gedai()
        gedai.fit_epochs(
            e_eeg,
            #duration             = GEDAI_DURATION,
            #overlap              = GEDAI_OVERLAP,
            #reject_by_annotation = GEDAI_REJECT_BY_ANNOT,
            reference_cov        = GEDAI_REFERENCE_COV,
            sensai_method        = GEDAI_SENSAI_METHOD,
            noise_multiplier     = GEDAI_NOISE_MULTIPLIER,
            verbose              = False,
        )

        fig = gedai.plot_fit()
        plt.show()

        e_trans_eeg = gedai.transform_epochs(
            e_eeg,
            #duration = GEDAI_DURATION,
            #overlap  = GEDAI_OVERLAP,
            verbose  = False,
        )

        # Reattach events, event_id, and metadata lost by transform_epochs()
        e_trans_eeg.events   = e_eeg.events.copy()
        e_trans_eeg.event_id = e_eeg.event_id.copy()
        if e_eeg.metadata is not None:
            e_trans_eeg.metadata = e_eeg.metadata.copy()

        # Reattach non-EEG channels
        if e_non_eeg is not None:
            with e_non_eeg.info._unlock():
                e_non_eeg.info["custom_ref_applied"] = e_trans_eeg.info["custom_ref_applied"]
            e_trans = e_trans_eeg.copy().add_channels([e_non_eeg])
        else:
            e_trans = e_trans_eeg

        output_dir = GEDAI_PATH / sub_name / ses_name / "eeg"
        output_dir.mkdir(parents=True, exist_ok=True)

        output_name = fif_path.name.replace("-epo.fif", "_desc-gedai-epo.fif")
        output_path = output_dir / output_name

        e_trans.save(output_path, overwrite=True)
        print(f"  Saved : {output_name}")

    except Exception as ex:
        print(f"  Error : {sub_name} / {ses_name} / {label} : {ex}")

    finally:
        for var in ["e", "e_eeg", "e_non_eeg", "gedai", "e_trans_eeg", "e_trans"]:
            if var in locals():
                del locals()[var]
        gc.collect()

Found 4 epoch files to process (EPOCH_TYPES='IC')

GEDAI : sub-01 / ses-01 / IC_left
  Saved : sub-01_ses-01_task-sidecut_IC_left_desc-gedai-epo.fif

GEDAI : sub-01 / ses-01 / IC_right


KeyboardInterrupt: 

In [ ]:
# -------------------------------------------------------------------------
# INSPECT GEDAI RESULT
# Compares the original file against the saved GEDAI output for one
# subject/session. Uses already-processed files — no refitting needed.
# -------------------------------------------------------------------------
wd          = Path.cwd()
EPOCHS_PATH = wd / "derivatives" / "03_epochs"
GEDAI_PATH  = wd / "derivatives" / "04a_epochs_gedai"

INSPECT_SUB   = "sub-01"
INSPECT_SES   = "ses-01"
INSPECT_LABEL = "IC_left" # IC_left, IC_right, RS_left, RS_right

orig_matches  = list((EPOCHS_PATH / INSPECT_SUB / INSPECT_SES / "eeg").glob(f"*{INSPECT_LABEL}-epo.fif"))
gedai_matches = list((GEDAI_PATH / INSPECT_SUB / INSPECT_SES / "eeg").glob(f"*{INSPECT_LABEL}_desc-gedai-epo.fif"))

if not orig_matches or not gedai_matches:
    print(f"Missing file(s) for {INSPECT_SUB} / {INSPECT_SES} / {INSPECT_LABEL} — "
          f"original: {len(orig_matches)}, GEDAI: {len(gedai_matches)}")
else:
    e_orig  = mne.read_epochs(orig_matches[0], preload=True)
    e_gedai = mne.read_epochs(gedai_matches[0], preload=True)

## 5.2. Spectral GEDAI

In [29]:
# -------------------------------------------------------------------------
# FUNCTION: trim_to_wavelet_length
# pywt.swt (used internally by GEDAI's MODWT) requires signal length to be
# divisible by 2**level. MNE epochs often have an odd sample count due to
# the inclusive tmax endpoint, so trim a few trailing samples if needed.
# -------------------------------------------------------------------------
def trim_to_wavelet_length(epochs, level):
    sfreq   = epochs.info["sfreq"]
    n_times = len(epochs.times)
    divisor = 2 ** level
    n_valid = (n_times // divisor) * divisor

    if n_valid == n_times:
        return epochs

    new_tmax = epochs.tmin + (n_valid - 1) / sfreq
    dropped  = n_times - n_valid
    print(f"  Trimming {dropped} trailing sample(s) for wavelet_level={level} compatibility")
    return epochs.crop(tmax=new_tmax)

In [ ]:
GEDAI_PATH          = wd / "derivatives" / "04a_epochs_gedai"
GEDAI_SPECTRAL_PATH = wd / "derivatives" / "04b_epochs_gedai_gedaiSpec"

# -------------------------------------------------------------------------
# CONFIGURATION
# Choose which epoch types to process.
# "IC"  — IC_left and IC_right only
# "RS"  — RS_left and RS_right only
# None  — all four epoch types
# -------------------------------------------------------------------------
EPOCH_TYPES = None  # "IC", "RS", or None to process all epochs

if EPOCH_TYPES == "IC":
    epoch_pattern = "*IC_*_desc-gedai-epo.fif"
elif EPOCH_TYPES == "RS":
    epoch_pattern = "*RS_*_desc-gedai-epo.fif"
elif EPOCH_TYPES is None:
    epoch_pattern = "*_desc-gedai-epo.fif"
else:
    raise ValueError(f"Unknown EPOCH_TYPES: '{EPOCH_TYPES}'. Choose 'IC', 'RS', or None.")

# -------------------------------------------------------------------------
# CONFIGURATION
# Spectral GEDAI: same algorithm as broadband GEDAI, but with a wavelet
# decomposition splitting the data into multiple frequency sub-bands.
# Each sub-band gets its own SENSAI threshold, allowing more targeted
# cleaning of high-frequency noise without touching low-frequency signal.
# wavelet_level=0 would be equivalent to the broadband version already run.
# Output: 04b_epochs_gedai_spectral/sub-XX/ses-XX/eeg/
# -------------------------------------------------------------------------
GEDAI_WAVELET_TYPE      = "haar"  # wavelet basis
GEDAI_WAVELET_LEVEL     = 6       # number of decomposition levels (sub-bands)
GEDAI_WAVELET_LOW_CUTOFF = None   # Hz — lowest frequency band boundary, or None for default
GEDAI_REFERENCE_COV     = "leadfield"
GEDAI_SENSAI_METHOD     = "gridsearch"
GEDAI_NOISE_MULTIPLIER  = 3.0
# -------------------------------------------------------------------------
gedai_spectral_files = sorted(GEDAI_PATH.rglob(epoch_pattern))
print(f"Found {len(gedai_spectral_files)} GEDAI-cleaned epoch files to process (EPOCH_TYPES='{EPOCH_TYPES}')")

for fif_path in gedai_spectral_files:
    ses_name = fif_path.parent.parent.name
    sub_name = fif_path.parent.parent.parent.name

    for candidate in ["IC_left", "IC_right", "RS_left", "RS_right"]:
        if candidate in fif_path.name:
            label = candidate
            break
    else:
        print(f"  Could not determine label from filename, skipping : {fif_path.name}")
        continue

    print(f"\nSpectral GEDAI : {sub_name} / {ses_name} / {label}")

    try:
        e = mne.read_epochs(fif_path, preload=True)

        e_eeg = e.copy().pick("eeg")
        non_eeg_picks = mne.pick_types(e.info, eeg=False, eog=True, emg=True, misc=True)
        e_non_eeg = e.copy().pick(non_eeg_picks) if len(non_eeg_picks) > 0 else None

        # Ensure sample count is compatible with the wavelet decomposition level
        e_eeg = trim_to_wavelet_length(e_eeg, GEDAI_WAVELET_LEVEL)
        if e_non_eeg is not None:
            e_non_eeg = trim_to_wavelet_length(e_non_eeg, GEDAI_WAVELET_LEVEL)

        gedai_spectral = Gedai(
            wavelet_type       = GEDAI_WAVELET_TYPE,
            wavelet_level      = GEDAI_WAVELET_LEVEL,
            wavelet_low_cutoff = GEDAI_WAVELET_LOW_CUTOFF,
        )
        gedai_spectral.fit_epochs(
            e_eeg,
            reference_cov    = GEDAI_REFERENCE_COV,
            sensai_method    = GEDAI_SENSAI_METHOD,
            noise_multiplier = GEDAI_NOISE_MULTIPLIER,
            verbose          = False,
        )

        fig = gedai_spectral.plot_fit()
        plt.show()

        e_trans_eeg = gedai_spectral.transform_epochs(
            e_eeg,
            verbose = False,
        )

        # Reattach events, event_id, and metadata lost by transform_epochs()
        e_trans_eeg.events   = e_eeg.events.copy()
        e_trans_eeg.event_id = e_eeg.event_id.copy()
        if e_eeg.metadata is not None:
            e_trans_eeg.metadata = e_eeg.metadata.copy()

        # Reattach non-EEG channels
        if e_non_eeg is not None:
            with e_non_eeg.info._unlock():
                e_non_eeg.info["custom_ref_applied"] = e_trans_eeg.info["custom_ref_applied"]
            e_trans = e_trans_eeg.copy().add_channels([e_non_eeg])
        else:
            e_trans = e_trans_eeg

        output_dir = GEDAI_SPECTRAL_PATH / sub_name / ses_name / "eeg"
        output_dir.mkdir(parents=True, exist_ok=True)

        output_name = fif_path.name.replace("_desc-gedai-epo.fif", "_desc-gedaiSpectral-epo.fif")
        output_path = output_dir / output_name

        e_trans.save(output_path, overwrite=True)
        print(f"  Saved : {output_name}")

    except Exception as ex:
        print(f"  Error : {sub_name} / {ses_name} / {label} : {ex}")

    finally:
        for var in ["e", "e_eeg", "e_non_eeg", "gedai_spectral", "e_trans_eeg", "e_trans"]:
            if var in locals():
                del locals()[var]
        gc.collect()

Found 4 GEDAI-cleaned epoch files to process (EPOCH_TYPES='IC')

Spectral GEDAI : sub-01 / ses-01 / IC_left
  Trimming 5 trailing sample(s) for wavelet_level=6 compatibility
  Trimming 5 trailing sample(s) for wavelet_level=6 compatibility


KeyboardInterrupt: 

In [ ]:
# -------------------------------------------------------------------------
# INSPECT SPECTRAL GEDAI RESULT
# Compares the original file against the saved GEDAI output for one
# subject/session. Uses already-processed files — no refitting needed.
# -------------------------------------------------------------------------
wd          = Path.cwd()
EPOCHS_PATH = wd / "derivatives" / "03_epochs"
GEDAI_PATH  = wd / "derivatives" / "04a_epochs_gedai"
SPEC_GEDAI_PATH = wd / "derivatives" / "04b_epochs_gedai_gedaiSpec"

INSPECT_SUB   = "sub-01"
INSPECT_SES   = "ses-01"
INSPECT_LABEL = "IC_left" # IC_left, IC_right, RS_left, RS_right

orig_matches  = list((EPOCHS_PATH / INSPECT_SUB / INSPECT_SES / "eeg").glob(f"*{INSPECT_LABEL}-epo.fif"))
gedai_matches = list((GEDAI_PATH / INSPECT_SUB / INSPECT_SES / "eeg").glob(f"*{INSPECT_LABEL}_desc-gedai-epo.fif"))
spec_gedai_matches = list((SPEC_GEDAI_PATH / INSPECT_SUB / INSPECT_SES / "eeg").glob(f"*{INSPECT_LABEL}_desc-gedaiSpectral-epo.fif"))

if not orig_matches or not gedai_matches or not spec_gedai_matches:
    print(f"Missing file(s) for {INSPECT_SUB} / {INSPECT_SES} / {INSPECT_LABEL} — "
          f"original: {len(orig_matches)}, GEDAI: {len(gedai_matches)}, Spectral GEDAI: {len(spec_gedai_matches)}")
else:
    e_orig  = mne.read_epochs(orig_matches[0], preload=True)
    e_gedai = mne.read_epochs(gedai_matches[0], preload=True)
    e_specgedai = mne.read_epochs(spec_gedai_matches[0], preload=True)

# 6. Adaptive Mixture Independent Component Analysis

Run the python adaptation of the adaptive mixture ICA (AMICA) [@palmerAMICAAdaptiveMixture2011].  

Link:  
[https://github.com/DerAndereJohannes/pyamica](https://github.com/DerAndereJohannes/pyamica)

## 6.1. AMICA fitting

In [ ]:
# -------------------------------------------------------------------------
# AMICA ICA FITTING
# Fits AMICA on epochs for each subject/session, using GEDAI-cleaned data.
# Rank is computed per subject to set n_components automatically.
# Fitted AMICA objects are saved to 05_amica_proc/
# -------------------------------------------------------------------------
wd = Path.cwd()
GEDAI_SPECTRAL_PATH = wd / "derivatives" / "04b_epochs_gedai_gedaiSpec"
AMICA_PROC_PATH = wd / "derivatives" / "05_gedai_gedaiSpec_amica_proc"

AMICA_FIT_HPASS = 3.0 # Hz - high-pass filtering for AMICA fitting

# -------------------------------------------------------------------------
# CONFIGURATION
# -------------------------------------------------------------------------
AMICA_EPOCH_TYPES = None  # "IC", "RS", or None to process all epochs

if AMICA_EPOCH_TYPES == "IC":
    epoch_pattern = "*IC_*-epo.fif"
elif AMICA_EPOCH_TYPES == "RS":
    epoch_pattern = "*RS_*-epo.fif"
elif AMICA_EPOCH_TYPES is None:
    epoch_pattern = "*-epo.fif"
else:
    raise ValueError(f"Unknown AMICA_EPOCH_TYPES: '{AMICA_EPOCH_TYPES}'. Choose 'IC', 'RS', or None.")

epoch_files = sorted(GEDAI_SPECTRAL_PATH.rglob(epoch_pattern))
print(f"Found {len(amica_files)} epoch files to process (AMICA_EPOCH_TYPES='{AMICA_EPOCH_TYPES}')")

for fif_path in epoch_files:
    ses_name = fif_path.parent.parent.name
    sub_name = fif_path.parent.parent.parent.name

    for candidate in ["IC_left", "IC_right", "RS_left", "RS_right"]:
        if candidate in fif_path.name:
            label = candidate
            break
    else:
        print(f"  Could not determine label from filename, skipping : {fif_path.name}")
        continue

    print(f"\nFitting AMICA : {sub_name} / {ses_name} / {label}")

    try:
        e = mne.read_epochs(fif_path, preload=True)
        e_fit = e.copy().filter(l_freq=AMICA_FIT_HPASS, h_freq=None, picks = "eeg")

        rank = mne.compute_rank(e_fit, tol=1e-6, tol_kind="relative")
        print(f"  Rank : {rank['eeg']}")

        amica = AmicaICA(
            n_models     = 1,
            n_components = rank["eeg"],
            do_reject    = True,
            reject_sigma = 3.0,
            num_reject   = 5,
            reject_int   = 1,
            device       = "cpu"
        )
        amica.fit(e_fit, picks="eeg")

        output_dir = AMICA_PROC_PATH / sub_name / ses_name / "eeg"
        output_dir.mkdir(parents=True, exist_ok=True)

        output_name = fif_path.name.replace("-epo.fif", "_amica")
        output_path = output_dir / output_name

        amica.save(str(output_path))
        print(f"  Saved AMICA : {output_name}")

    except Exception as ex:
        print(f"  Error : {sub_name} / {ses_name} / {label} : {ex}")

    finally:
        for var in ["e", "e_fit", "amica"]:
            if var in locals():
                del locals()[var]
        gc.collect()

Found 1 epoch files to process (AMICA_EPOCH_TYPES='None')

Fitting AMICA : sub-01 / ses-01 / IC_left
  Rank : 62
AMICA  T=144768  n_orig=63  M=1  J=3
  Rank deficiency detected: reducing from 63 to 62 components.
  After sphering: n=62  sldet=-50.7052
  Auto chunk_t=16384 (CPU, ~32 MB L3 target)
  Rejection 1/5 at iter 1: 2667/144768 samples excluded total (thresh=-191.9230)
  iter     1  lrate=1.000e-01  LL=-2.2715525165  nd=3.286e-01
  Rejection 2/5 at iter 2: 2739/144768 samples excluded total (thresh=-206.1097)
  Rejection 3/5 at iter 3: 2770/144768 samples excluded total (thresh=-209.3514)
  Rejection 4/5 at iter 4: 2792/144768 samples excluded total (thresh=-208.9040)
  Rejection 5/5 at iter 5: 2812/144768 samples excluded total (thresh=-208.6988)
  Starting Newton at iter 50 ...
  iter   100  lrate=1.000e+00  LL=-2.1479009570  nd=1.082e-02
  iter   200  lrate=1.000e+00  LL=-2.1436824814  nd=2.072e-03
  iter   300  lrate=1.000e+00  LL=-2.1430382393  nd=1.588e-03
  iter   400  lra

In [ ]:
# -------------------------------------------------------------------------
# OPTIONAL: INSPECT AMICA COMPONENTS
# Configure and run this block independently — it does not affect the
# main rejection loop. Set INSPECT = False to skip entirely.
# -------------------------------------------------------------------------
INSPECT       = True
AMICA_PROC_PATH = wd / "derivatives" / "05_gedai_gedaiSpec_amica_proc"
INSPECT_SUB   = "sub-01"
INSPECT_SES   = "ses-01"
INSPECT_LABEL = "IC_left"  # "IC_left", "IC_right", "RS_left", "RS_right"

if INSPECT:
    inspect_amica_file = sorted(AMICA_PROC_PATH.rglob(f"*{INSPECT_LABEL}*_amica.amica.npz"))
    inspect_amica_file = [f for f in inspect_amica_file
                          if INSPECT_SUB in f.parts and INSPECT_SES in f.parts]

    if len(inspect_amica_file) == 0:
        print(f"No AMICA file found for {INSPECT_SUB} / {INSPECT_SES} / {INSPECT_LABEL}")
    else:
        inspect_epo_name = inspect_amica_file[0].name.replace("_amica.amica.npz", "-epo.fif")
        inspect_epo_path = GEDAI_PATH / INSPECT_SUB / INSPECT_SES / "eeg" / inspect_epo_name

        if not inspect_epo_path.exists():
            print(f"Epoch file not found : {inspect_epo_path.name}")
        else:
            e_inspect   = mne.read_epochs(inspect_epo_path, preload=True)
            amica_insp  = AmicaICA.load(str(inspect_amica_file[0]))
            ica_inspect = amica_insp.to_mne_ica()

            ic_labels_insp = label_components(e_inspect, ica_inspect, method="iclabel")
            labs_insp  = ic_labels_insp["labels"]
            probs_insp = ic_labels_insp["y_pred_proba"]

            # -------------------------------------------------------------------------
            # PLOT COMPONENTS WITH ICLABEL PROBABILITIES
            # plot_components returns figure(s) — iterate axes and add label + prob
            # as a subtitle under each topography.
            # -------------------------------------------------------------------------
            figs = ica_inspect.plot_components(
                inst=e_inspect,
                title=f"{INSPECT_SUB} / {INSPECT_SES} / {INSPECT_LABEL}",
                show=False
            )

            # plot_components may return a single figure or a list
            if not isinstance(figs, list):
                figs = [figs]

            for fig in figs:
                for ax in fig.axes:
                    # Extract component index from axis title (e.g. "ICA000")
                    ax_title = ax.get_title()
                    if not ax_title.startswith("ICA"):
                        continue
                    try:
                        comp_idx = int(ax_title.replace("ICA", ""))
                        lab  = labs_insp[comp_idx]
                        prob = probs_insp[comp_idx].max()
                        ax.set_title(f"IC{comp_idx:02d}\n{lab}\n{prob:.2f}", fontsize=7)
                    except (ValueError, IndexError):
                        continue
                fig.tight_layout()
                fig.show()

            # -------------------------------------------------------------------------
            # PLOT SOURCES — click on time course to open detailed component view
            # -------------------------------------------------------------------------
            ica_inspect.plot_sources(e_inspect, block=True)

Loaded from /home/joel/Documents/BIDS - Kopie/CoMoCut_Proof-of-conept/derivatives_gedai/05_epochs_amica_proc/sub-01/ses-01/eeg/sub-01_ses-01_task-sidecut_IC_left_amica.amica.npz  (n_models=1, n_components=62)


## 6.2. AMICA Rejection
Label components with ICLabel and reject artifact components. 
In this study, we used a conservative approach retaining most of the signal by using a threshold of 85% for "eye blink", "eye movement" or "muscle artifact".

In [ ]:
# -------------------------------------------------------------------------
# AMICA COMPONENT REJECTION
# Loads fitted AMICA objects from 05_..._amica_proc/ and corresponding epochs
# from 04_.../.
# -------------------------------------------------------------------------
wd = Path.cwd()
GEDAI_SPECTRAL_PATH = wd / "derivatives" / "04b_epochs_gedai_gedaiSpec"
AMICA_PROC_PATH = wd / "derivatives" / "05_gedai_gedaiSpec_amica_proc"

# -------------------------------------------------------------------------
# CONFIGURATION
# AMICA_EPOCH_TYPES: "IC", "RS", or None (all epoch types)
# -------------------------------------------------------------------------
AMICA_EPOCH_TYPES = None    # "IC", "RS", or None

# -------------------------------------------------------------------------
# DISCOVER AMICA FILES BASED ON EPOCH TYPE SELECTION
# -------------------------------------------------------------------------
if AMICA_EPOCH_TYPES == "IC":
    amica_pattern = "*IC_*_amica.amica.npz"
elif AMICA_EPOCH_TYPES == "RS":
    amica_pattern = "*RS_*_amica.amica.npz"
elif AMICA_EPOCH_TYPES is None:
    amica_pattern = "*_amica.amica.npz"
else:
    raise ValueError(f"Unknown AMICA_EPOCH_TYPES: '{AMICA_EPOCH_TYPES}'. Choose 'IC', 'RS', or None.")

amica_files = sorted(AMICA_PROC_PATH.rglob(amica_pattern))
print(f"Found {len(amica_files)} fitted AMICA objects (AMICA_EPOCH_TYPES='{AMICA_EPOCH_TYPES}')")

Rejection mode : brain
Rejection tag  : brain50

Rejecting components : sub-01 / ses-01 / IC_left
Loaded from /home/joel/Documents/BIDS - Kopie/CoMoCut_Proof-of-conept/derivatives_gedai/05_gedai_spec_amica_proc/sub-01/ses-01/eeg/sub-01_ses-01_task-sidecut_IC_left_desc-gedaiSpectral_amica.amica.npz  (n_models=1, n_components=62)
  Excluding 46 / 62 components:
    IC00 : eye blink (0.99)
    IC01 : eye blink (1.00)
    IC03 : brain (0.45)
    IC04 : brain (0.50)
    IC05 : other (0.40)
    IC07 : other (0.64)
    IC11 : other (0.86)
    IC13 : muscle artifact (0.62)
    IC14 : brain (0.42)
    IC15 : muscle artifact (0.60)
    IC17 : other (0.55)
    IC18 : other (0.39)
    IC19 : other (0.94)
    IC20 : other (0.65)
    IC22 : other (0.51)
    IC24 : other (0.44)
    IC25 : other (0.83)
    IC26 : other (0.81)
    IC27 : other (0.79)
    IC28 : eye blink (0.67)
    IC29 : other (0.85)
    IC31 : other (0.58)
    IC32 : other (0.61)
    IC35 : muscle artifact (0.45)
    IC37 : other (0.

### 6.2.1. Soft rejection
Only reject IC with > 85 % artifact probability (eye, muscle)

In [ ]:
# -------------------------------------------------------------------------
# AMICA COMPONENT REJECTION
# Loads fitted AMICA objects from 05_amica_proc/ and corresponding epochs
# from 04_epochs_gedai/. Applies ICLabel classification and excludes
# components based on the selected rejection mode. Cleaned epochs are
# saved to 06_amica_rej/
# -------------------------------------------------------------------------

# --- Final bandpass filter (applied after AMICA component rejection) ---
FINAL_LFREQ = 1.0
FINAL_HFREQ = 40.0

# -------------------------------------------------------------------------
# CONFIGURATION
# Two rejection modes:
#   "category" — reject components whose top label matches a category
#                below AND whose confidence exceeds that category's
#                threshold (e.g. eye blink, muscle artifact)
#   "brain"    — keep only components whose top label is "brain" AND
#                confidence exceeds BRAIN_THRESHOLD; reject everything else
# -------------------------------------------------------------------------
REJECTION_MODE = "category"  # "category" or "brain"
REJ_TAG        = "rej85" # e.g. "brain50" or "rej85"

AMICA_REJ_PATH = wd / "derivatives" / f"06a_gedai_gedaiSpec_amica_{REJ_TAG}"


# --- Mode: "category" ---
EXCLUDE_CONFIG = {
    "eye blink"       : 0.85,
    "muscle artifact" : 0.85,
    "brain"           : None,
    "heart beat"      : None,
    "line noise"      : None,
    "channel noise"   : None,
    "other"           : None,
}

# --- Mode: "brain" ---
BRAIN_THRESHOLD = 0.50

print(f"Rejection mode : {REJECTION_MODE}")
print(f"Rejection tag  : {REJ_TAG}")

for amica_path in amica_files:
    ses_name = amica_path.parent.parent.name
    sub_name = amica_path.parent.parent.parent.name

    for candidate in ["IC_left", "IC_right", "RS_left", "RS_right"]:
        if candidate in amica_path.name:
            label = candidate
            break
    else:
        print(f"  Could not determine label from filename, skipping : {amica_path.name}")
        continue

    print(f"\nRejecting components : {sub_name} / {ses_name} / {label}")

    try:
        epo_name = amica_path.name.replace("_amica.amica.npz", "-epo.fif")
        epo_path = GEDAI_PATH / sub_name / ses_name / "eeg" / epo_name

        if not epo_path.exists():
            print(f"  Epoch file not found, skipping : {epo_path.name}")
            continue

        e = mne.read_epochs(epo_path, preload=True)

        amica   = AmicaICA.load(str(amica_path))
        ica_mne = amica.to_mne_ica()

        ic_labels = label_components(e, ica_mne, method="iclabel")
        labs  = ic_labels["labels"]
        probs = ic_labels["y_pred_proba"]  # top-label confidence per component

        # ---------------------------------------------------------------------
        # IDENTIFY COMPONENTS TO EXCLUDE, BASED ON SELECTED MODE
        # ---------------------------------------------------------------------
        if REJECTION_MODE == "category":
            exclude_idx = []
            for idx, (lab, p) in enumerate(zip(labs, probs)):
                threshold = EXCLUDE_CONFIG.get(lab)
                if threshold is not None and p > threshold:
                    exclude_idx.append(idx)

        elif REJECTION_MODE == "brain":
            exclude_idx = [
                idx for idx, (lab, p) in enumerate(zip(labs, probs))
                if not (lab == "brain" and p > BRAIN_THRESHOLD)
            ]

        else:
            raise ValueError(f"Unknown REJECTION_MODE: '{REJECTION_MODE}'. Choose 'category' or 'brain'.")

        print(f"  Excluding {len(exclude_idx)} / {len(labs)} components:")
        for idx in exclude_idx:
            print(f"    IC{idx:02d} : {labs[idx]} ({probs[idx]:.2f})")

        ep_rej          = e.copy()
        ica_mne.exclude = exclude_idx
        ica_mne.apply(ep_rej)

        # -----------------------------------------------------------------
        # FINAL BANDPASS FILTER
        # Applied after component rejection, so artifact components are
        # removed before any filtering — filtering first could smear
        # artifact energy into neighboring frequencies before AMICA/ICLabel
        # gets a chance to isolate it cleanly.
        # -----------------------------------------------------------------
        ep_rej.filter(l_freq=FINAL_LFREQ, h_freq=FINAL_HFREQ, picks="eeg")
        
        output_dir = AMICA_REJ_PATH / sub_name / ses_name / "eeg"
        output_dir.mkdir(parents=True, exist_ok=True)

        output_name = epo_name.replace("-epo.fif", f"_desc-amicaRej_{REJ_TAG}-epo.fif")
        output_path = output_dir / output_name

        ep_rej.save(output_path, overwrite=True)
        print(f"  Saved : {output_name}")

    except Exception as ex:
        print(f"  Error : {sub_name} / {ses_name} / {label} : {ex}")

    finally:
        for var in ["e", "amica", "ica_mne", "ep_rej"]:
            if var in locals():
                del locals()[var]
        gc.collect()

### 6.2.2. Hard rejection
Only keep ICs with brain probability > 50 %

In [ ]:
# -------------------------------------------------------------------------
# AMICA COMPONENT REJECTION
# Loads fitted AMICA objects from 05_amica_proc/ and corresponding epochs
# from 04_epochs_gedai/. Applies ICLabel classification and excludes
# components based on the selected rejection mode. Cleaned epochs are
# saved to 06_amica_rej/
# -------------------------------------------------------------------------

# --- Final bandpass filter (applied after AMICA component rejection) ---
FINAL_LFREQ = 1.0
FINAL_HFREQ = 40.0

# -------------------------------------------------------------------------
# CONFIGURATION
# Two rejection modes:
#   "category" — reject components whose top label matches a category
#                below AND whose confidence exceeds that category's
#                threshold (e.g. eye blink, muscle artifact)
#   "brain"    — keep only components whose top label is "brain" AND
#                confidence exceeds BRAIN_THRESHOLD; reject everything else
# -------------------------------------------------------------------------
REJECTION_MODE = "brain"  # "category" or "brain"
REJ_TAG        = "brain50" # e.g. "brain50" or "rej85"

AMICA_REJ_PATH = wd / "derivatives" / f"06b_gedai_gedaiSpec_amica_{REJ_TAG}"


# --- Mode: "category" ---
EXCLUDE_CONFIG = {
    "eye blink"       : 0.85,
    "muscle artifact" : 0.85,
    "brain"           : None,
    "heart beat"      : None,
    "line noise"      : None,
    "channel noise"   : None,
    "other"           : None,
}

# --- Mode: "brain" ---
BRAIN_THRESHOLD = 0.50

print(f"Rejection mode : {REJECTION_MODE}")
print(f"Rejection tag  : {REJ_TAG}")

for amica_path in amica_files:
    ses_name = amica_path.parent.parent.name
    sub_name = amica_path.parent.parent.parent.name

    for candidate in ["IC_left", "IC_right", "RS_left", "RS_right"]:
        if candidate in amica_path.name:
            label = candidate
            break
    else:
        print(f"  Could not determine label from filename, skipping : {amica_path.name}")
        continue

    print(f"\nRejecting components : {sub_name} / {ses_name} / {label}")

    try:
        epo_name = amica_path.name.replace("_amica.amica.npz", "-epo.fif")
        epo_path = GEDAI_PATH / sub_name / ses_name / "eeg" / epo_name

        if not epo_path.exists():
            print(f"  Epoch file not found, skipping : {epo_path.name}")
            continue

        e = mne.read_epochs(epo_path, preload=True)

        amica   = AmicaICA.load(str(amica_path))
        ica_mne = amica.to_mne_ica()

        ic_labels = label_components(e, ica_mne, method="iclabel")
        labs  = ic_labels["labels"]
        probs = ic_labels["y_pred_proba"]  # top-label confidence per component

        # ---------------------------------------------------------------------
        # IDENTIFY COMPONENTS TO EXCLUDE, BASED ON SELECTED MODE
        # ---------------------------------------------------------------------
        if REJECTION_MODE == "category":
            exclude_idx = []
            for idx, (lab, p) in enumerate(zip(labs, probs)):
                threshold = EXCLUDE_CONFIG.get(lab)
                if threshold is not None and p > threshold:
                    exclude_idx.append(idx)

        elif REJECTION_MODE == "brain":
            exclude_idx = [
                idx for idx, (lab, p) in enumerate(zip(labs, probs))
                if not (lab == "brain" and p > BRAIN_THRESHOLD)
            ]

        else:
            raise ValueError(f"Unknown REJECTION_MODE: '{REJECTION_MODE}'. Choose 'category' or 'brain'.")

        print(f"  Excluding {len(exclude_idx)} / {len(labs)} components:")
        for idx in exclude_idx:
            print(f"    IC{idx:02d} : {labs[idx]} ({probs[idx]:.2f})")

        ep_rej          = e.copy()
        ica_mne.exclude = exclude_idx
        ica_mne.apply(ep_rej)

        # -----------------------------------------------------------------
        # FINAL BANDPASS FILTER
        # Applied after component rejection, so artifact components are
        # removed before any filtering — filtering first could smear
        # artifact energy into neighboring frequencies before AMICA/ICLabel
        # gets a chance to isolate it cleanly.
        # -----------------------------------------------------------------
        ep_rej.filter(l_freq=FINAL_LFREQ, h_freq=FINAL_HFREQ, picks="eeg")
        
        output_dir = AMICA_REJ_PATH / sub_name / ses_name / "eeg"
        output_dir.mkdir(parents=True, exist_ok=True)

        output_name = epo_name.replace("-epo.fif", f"_desc-amicaRej_{REJ_TAG}-epo.fif")
        output_path = output_dir / output_name

        ep_rej.save(output_path, overwrite=True)
        print(f"  Saved : {output_name}")

    except Exception as ex:
        print(f"  Error : {sub_name} / {ses_name} / {label} : {ex}")

    finally:
        for var in ["e", "amica", "ica_mne", "ep_rej"]:
            if var in locals():
                del locals()[var]
        gc.collect()

# Summary

This notebook covers the preprocessing steps applied to the concatenated raw EEG data, preparing it for ICA-based artifact removal.

1. [Setting channel types and montage](#1-channel-types-montage) —
Non-EEG channels were assigned their correct types: `HEOG` and `VEOG` as EOG, `NeckEMG` as EMG, and `x_dir`, `y_dir`, `z_dir` as miscellaneous (acceleration sensors). The standard EasyCap M1 montage was applied to assign electrode locations.

2. [Filtering](#2-filter) —
A high-pass filter of 1 Hz was applied (together with the 131 Hz online low-pass filtering). Note that for future studies a 1 Hz high-pass only is recommended, leaving the low-pass cutoff (40 Hz) to be applied after full processing (after AMICA).

3. [Restoring the reference channel FCz](#3-restoring-reference-channel-and-average-reference) —
FCz was used as the online reference during recording and is therefore absent from the raw data. It was added back as a zero channel and the data was re-referenced to the average of all EEG channels.

4. [Epoching](#4-epoching) —
Epochs were created for direction cue (RS) and initial ground contact (IC) events, separately for left and right sidecuts, and ATR was attached as metadata. Processed files were saved to `derivatives/03_epochs/sub-XX/ses-XX/eeg/`.

5. [GEDAI](#5-gedai) —
The GEDAI pipeline was used for artifact rejection, applied at the epoch level. First the broadband ("normal") version was run, followed by the spectral version on top of its output, using a wavelet decomposition to allow more targeted, frequency-band-specific cleaning.

6. [Adaptive mixture independent component analysis (AMICA)](#6-adaptive-mixture-independent-component-analysis) —
AMICA was fitted per subject and session on the GEDAI-cleaned epochs to decompose the signal into independent components. Components were classified using ICLabel and rejected according to the configured criterion (category-based thresholds or brain-probability-only).

# Check: Event counts

In [7]:
def count_annotations(path):
    path = Path(path)

    fif_files = sorted(path.rglob("*.fif")) if path.is_dir() else [path]

    for fif_path in fif_files:
        ses_name = fif_path.parent.parent.name
        sub_name = fif_path.parent.parent.parent.name

        raw = mne.io.read_raw_fif(fif_path, preload=False, verbose=False)

        ann_counts = defaultdict(int)
        for ann in raw.annotations:
            ann_counts[ann["description"]] += 1

        print(f"\n  {sub_name} / {ses_name}")
        for desc, count in sorted(ann_counts.items()):
            print(f"    {desc:<20} : {count}")

In [6]:
wd = Path.cwd()
PYPREP_PATH = wd / "derivatives" / "04_pyprep"
ASR_PATH    = wd / "derivatives" / "05_asr"
CONCAT_RAW_PATH = wd / "derivatives" / "02_concat-raw"
FILT_REF_PATH = wd / "derivatives" / "03_filt-ref"

In [10]:
count_annotations(ASR_PATH)


  sub-01 / ses-01
    BAD boundary         : 3
    EDGE boundary        : 3
    IC_left              : 58
    IC_right             : 65
    RS_left              : 58
    RS_right             : 65

  sub-02 / ses-01
    BAD boundary         : 2
    EDGE boundary        : 2
    IC_left              : 42
    IC_right             : 51
    RS_left              : 42
    RS_right             : 51


# Check data rank:

In [9]:
wd = Path.cwd()
PYPREP_PATH = wd / "derivatives" / "04_pyprep"
ASR_PATH    = wd / "derivatives" / "05_asr"
CONCAT_RAW_PATH = wd / "derivatives" / "02_concat-raw"
FILT_REF_PATH = wd / "derivatives" / "03_filt-ref"

In [10]:
# -------------------------------------------------------------------------
# FUNCTION: check_rank
# Loads .fif files from a given path and prints the data rank per file.
# Accepts a directory (searches recursively) or a single .fif file.
#
# Parameters
# ----------
# path    : Path — directory or single .fif file
# picks   : str  — channel type to compute rank for (default: "eeg")
# -------------------------------------------------------------------------
def check_rank(path, picks="eeg"):
    path = Path(path)

    fif_files = sorted(path.rglob("*.fif")) if path.is_dir() else [path]

    for fif_path in fif_files:
        ses_name = fif_path.parent.parent.name
        sub_name = fif_path.parent.parent.parent.name

        try:
            # Read epochs or raw depending on file type
            if "-epo.fif" in fif_path.name:
                data = mne.read_epochs(fif_path, preload=True, verbose=False)
            else:
                data = mne.io.read_raw_fif(fif_path, preload=True, verbose=False)

            rank = mne.compute_rank(data, tol=1e-6, tol_kind="relative")
            print(f"  {sub_name} / {ses_name} / {fif_path.name}")
            print(f"    {picks} rank : {rank.get(picks, 'n/a')}")

        except Exception as e:
            print(f"  Error : {sub_name} / {ses_name} : {e}")

In [11]:
check_rank(PYPREP_PATH)

  sub-01 / ses-01 / sub-01_ses-01_task-sidecut_desc-pyprep_eeg.fif
    eeg rank : 56
  sub-02 / ses-01 / sub-02_ses-01_task-sidecut_desc-pyprep_eeg.fif
    eeg rank : 61


In [19]:
check_rank(ASR_PATH)

  sub-01 / ses-01 / sub-01_ses-01_task-sidecut_desc-asr_eeg.fif
    eeg rank : 56
  sub-01 / ses-01 / sub-01_ses-01_task-sidecut_desc-rasr_eeg.fif
    eeg rank : 22
  sub-02 / ses-01 / sub-02_ses-01_task-sidecut_desc-asr_eeg.fif
    eeg rank : 61
  sub-02 / ses-01 / sub-02_ses-01_task-sidecut_desc-rasr_eeg.fif
    eeg rank : 22
